In [1]:
import pandas as pd
import ollama,chromadb
import uuid

In [2]:
recepies_1 = pd.read_csv('data/recipes_1.csv')


In [3]:
recepies_1.dropna(inplace=True)
recepies_1 = recepies_1[:2000]

In [4]:
import ast
recipes_df = pd.DataFrame()
recipes_df['recipes_name'] = recepies_1['name']
recipes_df['description'] = recepies_1['description']
recipes_df['ingredients'] = recepies_1['ingredients'].apply(ast.literal_eval)
recipes_df['nutrition'] = recepies_1['nutrition'].apply(ast.literal_eval)
recipes_df['directions'] = recepies_1['steps'].apply(ast.literal_eval)
recipes_df['no_of_steps'] = recepies_1['n_steps']
recipes_df['total_time'] = recepies_1['minutes']


In [ ]:
from chromadb.utils.embedding_functions import OllamaEmbeddingFunction
EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'


client = chromadb.PersistentClient(path="./cooking_recipes_app")
collection = client.get_or_create_collection(name="recipes_data",embedding_function=OllamaEmbeddingFunction(model_name=EMBEDDING_MODEL,url="http://localhost:11434/api/embeddings"),metadata={"hnsw:space": "cosine"})


In [6]:
import uuid
import ollama
from concurrent.futures import ThreadPoolExecutor, as_completed


def ingest_data_to_chroma(df, collection, model_name, batch_size=100, max_workers=4):
    """
    Parallelizes the embedding and ingestion of a DataFrame into ChromaDB.
    """
    
    def process_batch(batch_df):
        records = batch_df.to_dict('records')
        texts = [
            f"Recipe: {r['recipes_name']}. Ingredients: {r['ingredients']}." 
            for r in records
        ]
        response = ollama.embed(model=model_name, input=texts)
        embeddings = response['embeddings']       
        metadatas = [{
            'ingredients': list(r.get('ingredients', '')),
            "description": str(r.get('description', '')),
            "name": str(r.get('recipes_name', '')),
            "time": int(r.get('total_time', 0)),
            "steps": list(r.get('directions', '')),
            "steps_count": int(r.get('no_of_steps', 0)),
            "nutrition": list(r.get('nutrition', ''))
        } for r in records]
        
        ids = [str(uuid.uuid4()) for _ in range(len(records))]
        collection.add(
            ids=ids,
            documents=texts,
            embeddings=embeddings,
            metadatas=metadatas
        )

    size = len(df)
    batches = [df.iloc[i : i + batch_size] for i in range(0, size, batch_size)]
    total_batches = len(batches)

    print(f"🚀 Starting parallel ingestion: {size} rows | {total_batches} batches...")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(process_batch, b): i for i, b in enumerate(batches)}
        
        for future in as_completed(futures):
            idx = futures[future]
            try:
                future.result()
                print(f"✅ Batch {idx + 1}/{total_batches} indexed.")
            except Exception as e:
                print(f"❌ Batch {idx + 1} failed: {e}")

    print(f"🎉 Ingestion Complete!")

ingest_data_to_chroma(
    df=recipes_df, 
    collection=collection, 
    model_name=EMBEDDING_MODEL, 
    batch_size=500, 
    max_workers=4
)

🚀 Starting parallel ingestion: 2000 rows | 4 batches...
✅ Batch 2/4 indexed.
✅ Batch 1/4 indexed.
✅ Batch 3/4 indexed.
✅ Batch 4/4 indexed.
🎉 Ingestion Complete!


In [7]:
res = collection.query(
    query_texts=["What are some quick recipes with chicken,butter,salt,soy sauce?"],
    n_results=5,
    include=['metadatas']
)

In [8]:
res

{'ids': [['215fcfae-d345-49a2-b19e-9e4f8b9b6c7a',
   '74528fd6-adde-4094-a6b5-bc6bc164206d',
   '9af3ecc0-bffb-487c-8bda-7a8209e083a8',
   '2938c923-a72a-4fca-a9f3-e36286b37ec1',
   'e4fd6e4a-6819-4cc0-a9be-8ecd1ea58a56']],
 'embeddings': None,
 'documents': None,
 'uris': None,
 'included': ['metadatas'],
 'data': None,
 'metadatas': [[{'steps': ['mix together all the spices and toss through the chicken',
     'melt butter in skillet over medium high heat',
     'add onion and garlic and cook until soft',
     'add chicken',
     'stir until golden',
     'add tomato paste and garam masala',
     'stir in cream',
     'season and simmer for 5 - 8 minutes until rich and creamy'],
    'time': 10,
    'steps_count': 8,
    'nutrition': [471.9, 59.0, 9.0, 10.0, 49.0, 88.0, 2.0],
    'description': "a very quick, easy and oh so tasy chicken dish that you can prepare in the shortest time.  by the timeyou've cooked your rice, this is ready and waiting to be gobbled up :)\r\n\r\ndon't be fuss